This script is used to:
1. Check for gaps in the non-median classified CD rasters vs the median classified CD rasters in Dhruvi's work
2. Check how many conflict pixels exist
3. Resolve conflicts based on output of model with best F1-score
4. Export these resolved images as a 3-class (High, Medium, Low) CD raster for each ACZ

# Set up

In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='ee-mtpictd')

# Config

In [2]:
err = ee.ErrorMargin(100)
scale = 100
year = 2022

india = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'India'))
kolar = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2').filter(ee.Filter.eq("ADM2_NAME","Kolar"))
karnataka = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', "Karnataka")).first()

indiaDistricts = ee.FeatureCollection("projects/ee-indiasat/assets/india_district_boundaries")
indiaACZs = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")

In [3]:
aczList = [
  'Western Himalayan Region',
  'Eastern Himalayan Region',
  'Lower Gangetic Plain Region',
  'Middle Gangetic Plain Region',
  'Upper Gangetic Plain Region',
  'Trans Gangetic Plain Region',
  'Eastern Plateau & Hills Region',
  'Central Plateau & Hills Region',
  'Western Plateau and Hills Region',
  'Southern Plateau and Hills Region',
  'East Coast Plains & Hills Region'
]

aczAcronymDict = {'Eastern Plateau & Hills Region': 'EPAHR',
                  'Southern Plateau and Hills Region': 'SPAHR',
                  'East Coast Plains & Hills Region': 'ECPHR',
                  'Western Plateau and Hills Region': 'WPAHR',
                  'Central Plateau & Hills Region': 'CPAHR',
                  'Lower Gangetic Plain Region': 'LGPR',
                  'Middle Gangetic Plain Region': 'MGPR',
                  'Eastern Himalayan Region': 'EHR',
                  'Western Himalayan Region': 'WHR',
                  'Upper Gangetic Plain Region': 'UGPR',
                  'Trans Gangetic Plain Region': 'TGPR',
                  'West Coast Plains & Ghat Region': 'WCPGR',
                  'Gujarat Plains & Hills Region': 'GPHR',
                  'Western Dry Region': 'WDR'}

# In percentage (not percentile)
percentileThresholdDict = {
  'Western Himalayan Region': 50,
  'Eastern Himalayan Region': 50,
  'Lower Gangetic Plain Region': 67,
  'Middle Gangetic Plain Region': 75,
  'Upper Gangetic Plain Region': 40,
  'Trans Gangetic Plain Region': 83,
  'Eastern Plateau & Hills Region': 83,
  'Central Plateau & Hills Region': 72,
  'Western Plateau and Hills Region': 80,
  'Southern Plateau and Hills Region': 83,
  'East Coast Plains & Hills Region': 40
}

medianThresholdDict = {
  'Western Himalayan Region': 78,
  'Eastern Himalayan Region': 86,
  'Lower Gangetic Plain Region': 48,
  'Middle Gangetic Plain Region': 50,
  'Upper Gangetic Plain Region': 67,
  'Trans Gangetic Plain Region': 55,
  'Eastern Plateau & Hills Region': 60,
  'Central Plateau & Hills Region': 51,
  'Western Plateau and Hills Region': 57,
  'Southern Plateau and Hills Region': 62,
  'East Coast Plains & Hills Region': 67
}


aczNumFilesDict = {
  'Western Himalayan Region': 5,
  'Eastern Himalayan Region': 3,
  'Lower Gangetic Plain Region': 2,
  'Middle Gangetic Plain Region': 1,
  'Upper Gangetic Plain Region': 2,
  'Trans Gangetic Plain Region': 1,
  'Eastern Plateau & Hills Region': 17,
  'Central Plateau & Hills Region': 6,
  'Western Plateau and Hills Region': 4,
  'Southern Plateau and Hills Region': 5,
  'East Coast Plains & Hills Region': 4
}

aczMedianNumFilesDict = {
  'Western Himalayan Region': 2,
  'Eastern Himalayan Region': 2,
  'Lower Gangetic Plain Region': 1,
  'Middle Gangetic Plain Region': 1,
  'Upper Gangetic Plain Region': 1,
  'Trans Gangetic Plain Region': 1,
  'Eastern Plateau & Hills Region': 5,
  'Central Plateau & Hills Region': 2,
  'Western Plateau and Hills Region': 1,
  'Southern Plateau and Hills Region': 2,
  'East Coast Plains & Hills Region': 2
}

aczBestModelDict = {
  'Western Himalayan Region': 'percentile',
  'Eastern Himalayan Region': 'percentile',
  'Lower Gangetic Plain Region': 'median',
  'Middle Gangetic Plain Region': 'median',
  'Upper Gangetic Plain Region': 'percentile',
  'Trans Gangetic Plain Region': 'median',
  'Eastern Plateau & Hills Region': 'median',
  'Central Plateau & Hills Region': 'median',
  'Western Plateau and Hills Region': 'median',
  'Southern Plateau and Hills Region': 'median',
  'East Coast Plains & Hills Region': 'median'
}

# Utils

In [4]:
def get_img_collection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})['assets']
  image_list = [ee.Image(asset['name']) for asset in asset_list]
  return ee.ImageCollection(image_list)


# Checking conflicts

## Get ACZ images (median and percentile)

In [5]:
def getAczImage(acz, median):
  acronym = aczAcronymDict[acz]
  imgList = []
  if median:
    n = aczMedianNumFilesDict[acz]
  else:
    n = aczNumFilesDict[acz]
  for i in range(n):
    if median:
      if acronym == 'EPAHR' or acronym == 'LGPR':
        path = 'projects/ee-mtpictd/assets/harsh/ccd_2022/cc_2022_result_'
      else:
        path = 'projects/ee-mtpictd/assets/harsh/ccd_2022/ccd_2022_result_'

      path += str(i) + '_' + acronym
    else:
      path = 'projects/ee-mtpictd/assets/anoushka/cd_rasters_2022/'
      path += acronym + '_' + str(i)
    imgList.append(ee.Image(path))
  return ee.ImageCollection.fromImages(imgList).mosaic()


In [6]:
roi = indiaACZs

percentileImgList = []
medianImgList = []
for acz in aczList:
  acz_roi = ee.Feature(indiaACZs.filter(ee.Filter.eq('regionname', acz)).first())
  percentileImg = getAczImage(acz, median=False).clip(acz_roi.geometry())
  medianImg = getAczImage(acz, median=True).clip(acz_roi.geometry())
  percentileImgList.append(percentileImg)
  medianImgList.append(medianImg)

percentileImg = ee.ImageCollection.fromImages(percentileImgList).mosaic().clip(roi.geometry())
medianImg = ee.ImageCollection.fromImages(medianImgList).mosaic().clip(roi.geometry())

In [7]:
Map = geemap.Map()
Map.addLayer(medianImg.select('cc'), {'min':0, 'max':1, 'palette' : ['red', 'green']}, 'Canopy Density')
Map.addLayer(indiaACZs, {}, 'ACZs')
Map.centerObject(indiaACZs, 4)

Map

EEException: Description length exceeds maximum.

## Checking gaps in percentile image

Some areas around Bihar and in the Northeast have gaps in percentile image, otherwise the two images match up everywhere else

In [ ]:
mask1 = medianImg.select('cc').mask().clip(roi.geometry()) # Where image1 has data
mask2 = percentileImg.select('cc').mask().clip(roi.geometry())  # Where image2 has data

diff = mask1.subtract(mask2) # 1 where median and not percentile, 0 where both, -1 where percentile and not median

## Checking conflicts

In [8]:
import pprint

def checkConflicts(acz, verbose=True):
  acz_roi = ee.Feature(indiaACZs.filter(ee.Filter.eq('regionname', acz)).first())
  percentileImg = getAczImage(acz, median=False).clip(acz_roi.geometry()).select('cc')
  medianImg = getAczImage(acz, median=True).clip(acz_roi.geometry()).select('cc')

  # Mask out pixels where either is null
  mask = percentileImg.mask().And(medianImg.mask())
  percentileImg = percentileImg.updateMask(mask).rename('cc_percentile')
  medianImg = medianImg.updateMask(mask).rename('cc_median')

  classified = (
      percentileImg.eq(0).And(medianImg.eq(0)).multiply(0)
      .add(percentileImg.eq(0).And(medianImg.eq(1)).multiply(1))
      .add(percentileImg.eq(1).And(medianImg.eq(0)).multiply(2))
      .add(percentileImg.eq(1).And(medianImg.eq(1)).multiply(3))
  ).clip(acz_roi.geometry()).rename('cc_conflicts')

  if verbose:
    counts_dict = classified.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=acz_roi.geometry(),
        scale=25,
        maxPixels=1e10
    ).getInfo()

    counts = counts_dict['cc_conflicts']
    counts_rounded = {int(k): round(v) for k, v in counts.items()}
    total = sum(counts_rounded.values())

    matrix = [
        [counts_rounded.get(0, 0), counts_rounded.get(1, 0)],
        [counts_rounded.get(2, 0), counts_rounded.get(3, 0)]
    ]

    print(acz)
    print(f'Median value: {medianThresholdDict[acz]}%     Percentile value: {percentileThresholdDict[acz]}%')
    print("Pixel value counts:")

    def format_cell(val, total):
        percent = val / total * 100
        return f"{val:,} ({percent:5.2f}%)"

    print(f"{'':16} | {'Median Low':>20} | {'Median High':>20}")
    print('-' * 64)

    row_labels = ['Percentile Low', 'Percentile High']
    for i, row in enumerate(matrix):
        formatted = [format_cell(val, total) for val in row]
        print(f"{row_labels[i]:16} | {formatted[0]:>20} | {formatted[1]:>20}")

    print('-' * 64)
    print(f"{'Total pixels':16}: {total:,}")
    print()

  return classified.addBands(percentileImg).addBands(medianImg)


In [9]:
acz = 'Lower Gangetic Plain Region'
conflictImg = checkConflicts(acz).set('acz', acz)
print(conflictImg.getInfo())

# conflictImgList = []
# for acz in aczList:
#   if acz == 'Western Himalayan Region':
#     continue
#   conflictImg = checkConflicts(acz, verbose=False).set('acz', acz)
#   conflictImgList.append(conflictImg)

# conflictImgs = ee.ImageCollection.fromImages(conflictImgList)

Output hidden; open in https://colab.research.google.com to view.

## Visualization

In [ ]:
ccd_vis = {
  'min' : 0,
  'max' : 1,
  'palette' : ['red', 'green']
}

mask_vis = {
  'min' : 0,
  'max' : 1,
  'palette' : ['red', 'green']
}

conflict_vis_percentile_low = {
  'min' : 0,
  'max' : 3,
  'palette' : ['blue', 'red', 'yellow', 'green']
}

legend_low = [
    ('P low, M low', 'blue'),
    ('P low, M high', 'red'),
    ('P high, M low', 'yellow'),
    ('P high, M high', 'green')
]


conflict_vis_percentile_high = {
  'min' : 0,
  'max' : 3,
  'palette' : ['blue', 'yellow', 'red', 'green']
}

legend_high = [
    ('P low, M low', 'blue'),
    ('P low, M high', 'yellow'),
    ('P high, M low', 'red'),
    ('P high, M high', 'green')
]

resolved_vis = {
  'min' : 0,
  'max' : 2,
  'palette' : ['blue', 'yellow', 'green']
}

In [ ]:
def addConflictMaps(acz, map):
  if acz == 'Western Himalayan Region':
    return
  img = conflictImgs.filter(ee.Filter.eq('acz', acz)).first()
  if percentileThresholdDict[acz] < medianThresholdDict[acz]:
    map.addLayer(img.select('cc_conflicts'), conflict_vis_percentile_low, f'{aczAcronymDict[acz]} conflicts')
    # map.add_legend(
    #     title='Pixel Categories',
    #     legend_colors=legend_low
    # )
  else:
    map.addLayer(img.select('cc_conflicts'), conflict_vis_percentile_high, f'{aczAcronymDict[acz]} conflicts')
    # map.add_legend(
    #     title='Pixel Categories',
    #     legend_colors=legend_high
    # )
  # map.addLayer(img.select('cc_percentile'), ccd_vis, f'{aczAcronymDict[acz]} Percentile')
  # map.addLayer(img.select('cc_median'), ccd_vis, f'{aczAcronymDict[acz]} Median')


In [ ]:
Map = geemap.Map()
# acz = 'Trans Gangetic Plain Region'
# roi = indiaACZs.filter(ee.Filter.eq('regionname', acz)).first()
# Map.addLayer(percentileImg.select('cc'), ccd_vis, 'Percentile')
# Map.addLayer(medianImg.select('cc'), ccd_vis, 'Median')
# Map.addLayer(diff, {'min' : -1, 'max' : 1, 'palette' : ['red', 'white', 'green']}, 'Diff Image')
# Map.addLayer(only_in_image2, {'palette' : ['red']}, 'Only in percentile')
# Map.addLayer(roi, {}, 'ROI')
# for acz in aczList:
#   addConflictMaps(acz, Map)
Map.addLayer(conflictImg.select('cc_conflicts'), conflict_vis_percentile_low, f'{aczAcronymDict[acz]} conflicts')
Map.centerObject(indiaACZs, 4)

Map

NameError: name 'conflictImg' is not defined

## Resolve conflicts and export

In [ ]:
def resolveConflicts(acz):
  acz_roi = ee.Feature(indiaACZs.filter(ee.Filter.eq('regionname', acz)).first())
  percentileImg = getAczImage(acz, median=False).clip(acz_roi.geometry()).select('cc')
  medianImg = getAczImage(acz, median=True).clip(acz_roi.geometry()).select('cc')

  # Mask out pixels where either is null
  mask = percentileImg.mask().And(medianImg.mask())
  percentileImg = percentileImg.updateMask(mask).rename('cc_percentile')
  medianImg = medianImg.updateMask(mask).rename('cc_median')

  if aczBestModelDict[acz] == 'percentile':
    if percentileThresholdDict[acz] < medianThresholdDict[acz]:
      resolved = (
          percentileImg.eq(0).multiply(0)
          .add(percentileImg.eq(1).And(medianImg.eq(0)).multiply(1))
          .add(percentileImg.eq(1).And(medianImg.eq(1)).multiply(2))
      ).clip(acz_roi.geometry()).rename('cc_final')
    else:
      resolved = (
          percentileImg.eq(0).And(medianImg.eq(0)).multiply(0)
          .add(percentileImg.eq(0).And(medianImg.eq(1)).multiply(1))
          .add(percentileImg.eq(1).multiply(2))
      ).clip(acz_roi.geometry()).rename('cc_final')
  else:
    if percentileThresholdDict[acz] < medianThresholdDict[acz]:
      resolved = (
          percentileImg.eq(0).And(medianImg.eq(0)).multiply(0)
          .add(percentileImg.eq(1).And(medianImg.eq(0)).multiply(1))
          .add(medianImg.eq(1).multiply(2))
      ).clip(acz_roi.geometry()).rename('cc_final')
    else:
      resolved = (
          medianImg.eq(0).multiply(0)
          .add(percentileImg.eq(0).And(medianImg.eq(1)).multiply(1))
          .add(percentileImg.eq(1).And(medianImg.eq(1)).multiply(2))
      ).clip(acz_roi.geometry()).rename('cc_final')

  return resolved


Check pipeline for 1 ACZ

In [ ]:
acz = 'Southern Plateau and Hills Region'
conflictImg = checkConflicts(acz, verbose=False).set('acz', acz)
print(conflictImg.bandNames().getInfo())

['cc_conflicts', 'cc_percentile', 'cc_median']


In [ ]:
resolvedImg = resolveConflicts(acz)
print(resolvedImg.getInfo())

{'type': 'Image', 'bands': [{'id': 'cc_final', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 3}, 'dimensions': [8, 11], 'origin': [74, 9], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:footprint': {'type': 'Polygon', 'coordinates': [[[74.08911405080786, 15.67300075611287], [74.0906164144964, 15.668122018730983], [74.09715836372015, 15.661358831397594], [74.10470505010736, 15.657196772525282], [74.11407128783293, 15.655840760888953], [74.11876789686367, 15.652459828676376], [74.1217307387772, 15.652407537373454], [74.13560145546965, 15.649283585123953], [74.13969236675383, 15.650839850636334], [74.1478576138201, 15.651948328134356], [74.15615850401663, 15.650764283951155], [74.15887222799253, 15.654562144586894], [74.16639778930423, 15.6542300113627], [74.16840703870643, 15.672329111921082], [74.17165558384178, 15.673592380691696], [74.17601946706799, 15.679518296447604], [74.1831763428727, 15.683144997377862], [74.1907624564

In [ ]:
Map = geemap.Map()
Map.addLayer(conflictImg.select('cc_percentile'), ccd_vis, 'Percentile')
Map.addLayer(conflictImg.select('cc_median'), ccd_vis, 'Median')
Map.addLayer(conflictImg.select('cc_conflicts'), conflict_vis_percentile_high, 'Conflict')
Map.addLayer(resolvedImg, resolved_vis, 'Resolved')

Map.centerObject(indiaACZs, 4)
Map

Map(center=[23.22103836473429, 79.4609888487962], controls=(WidgetControl(options=['position', 'transparent_bg…

Export resolved image for each ACZ

In [ ]:
for acz in aczList:
  acz_roi = indiaACZs.filter(ee.Filter.eq('regionname', acz)).first()
  img = resolveConflicts(acz)
  acronym = aczAcronymDict[acz]

  task = ee.batch.Export.image.toAsset(
    image=img,
    description=f'Conflict_resolved_CCD_{acronym}',
    assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/final_cd_rasters_2022/{acronym}',
    scale = 25,
    maxPixels=1e13
  )
  task.start()
  print(f'Export started for {acronym}')


Export started for WHR
Export started for EHR
Export started for LGPR
Export started for MGPR
Export started for UGPR
Export started for TGPR
Export started for EPAHR
Export started for CPAHR
Export started for WPAHR
Export started for SPAHR
Export started for ECPHR


In [ ]:
final_ccd = get_img_collection('projects/ee-ictd-dhruvi/assets/anoushka/final_cd_rasters_2022').mosaic().clip(indiaACZs.geometry())